# 192. SWE-bench：代码 Agent 的沙箱、Patch 与测试判分怎样实现？

> **面试问题：怎样把 observe、patch、test、revision gate 和最小变更约束做成可复放代码 Agent loop？**

## 先给结论

高质量回答需要同时说明目标状态、可执行策略、状态版本、失败回滚和可复放评测。下面不用 Agent 框架或远程工具，而是用受控的内存模型把核心合同写出来；断言只证明教学实现的不变量，不能替代真实服务的隔离、审计、权限与压测。

## 一手资料

- [SWE-bench](https://arxiv.org/abs/2310.06770)
- [SWE-agent](https://arxiv.org/abs/2405.15793)
- [SWE-bench Verified](https://www.swebench.com/verified.html)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "explicit-assertions", "production": "needs-isolation"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "explicit-assertions"  # 执行本行的状态、计算或校验逻辑。
assert "isolation" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：代码 Agent 的答案为什么必须在沙箱中判分

代码修复不是生成一段 patch 文本，而是让特定 revision 的仓库通过目标测试，同时不破坏无关行为。教学 sandbox 保存在内存中，避免访问真实文件系统；真实系统还必须固定依赖、网络、时限、基线 commit 和隐藏测试。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class Patch:  # 执行本行的状态、计算或校验逻辑。
    path: str  # 执行本行的状态、计算或校验逻辑。
    before: str  # 执行本行的状态、计算或校验逻辑。
    after: str  # 执行本行的状态、计算或校验逻辑。
base_files = {"calc.py": "def score(a, b):\n    return a + b\n"}  # 执行本行的状态、计算或校验逻辑。
assert "return a + b" in base_files["calc.py"]  # 执行本行的状态、计算或校验逻辑。
assert len(base_files) == 1  # 执行本行的状态、计算或校验逻辑。
assert base_files["calc.py"].startswith("def score")  # 执行本行的状态、计算或校验逻辑。


## 2. 沙箱状态：revision 是 patch 的前置条件

Agent 观察到的代码版本和提交 patch 的版本必须一致，否则并发修改会使补丁语义漂移。`apply` 同时检查 revision、目标文件和 before 内容；这一层相当于小型乐观锁，也让每次尝试可以复放。


In [ ]:
class RepoSandbox:  # 执行本行的状态、计算或校验逻辑。
    def __init__(self, files):  # 执行本行的状态、计算或校验逻辑。
        self.files = dict(files)  # 执行本行的状态、计算或校验逻辑。
        self.revision = 0  # 执行本行的状态、计算或校验逻辑。
        self.history = []  # 执行本行的状态、计算或校验逻辑。
    def inspect(self, path):  # 执行本行的状态、计算或校验逻辑。
        return self.files[path], self.revision  # 执行本行的状态、计算或校验逻辑。
    def apply(self, patch, expected_revision):  # 执行本行的状态、计算或校验逻辑。
        if expected_revision != self.revision or self.files.get(patch.path) != patch.before:  # 执行本行的状态、计算或校验逻辑。
            raise ValueError("补丁基线已变化")  # 执行本行的状态、计算或校验逻辑。
        self.files[patch.path] = patch.after  # 执行本行的状态、计算或校验逻辑。
        self.revision += 1  # 执行本行的状态、计算或校验逻辑。
        self.history.append(patch.path)  # 执行本行的状态、计算或校验逻辑。
sandbox = RepoSandbox(base_files)  # 执行本行的状态、计算或校验逻辑。
source, revision = sandbox.inspect("calc.py")  # 执行本行的状态、计算或校验逻辑。
assert revision == 0  # 执行本行的状态、计算或校验逻辑。
assert source == base_files["calc.py"]  # 执行本行的状态、计算或校验逻辑。


## 3. 可执行 oracle：先写失败测试，再评估 patch

SWE-bench 的核心是以测试验证修复。演示只执行固定、可信的单文件字符串，且移除 builtins；不要对不受信任仓库直接 `exec`。生产系统应该在容器/VM 内运行依赖安装、测试命令和资源限制。


In [ ]:
def run_tests(files):  # 执行本行的状态、计算或校验逻辑。
    namespace = {"__builtins__": {}}  # 执行本行的状态、计算或校验逻辑。
    exec(files["calc.py"], namespace)  # 执行本行的状态、计算或校验逻辑。
    score = namespace["score"]  # 执行本行的状态、计算或校验逻辑。
    return {"subtract": score(7, 2) == 5, "zero": score(3, 3) == 0}  # 执行本行的状态、计算或校验逻辑。
baseline_tests = run_tests(sandbox.files)  # 执行本行的状态、计算或校验逻辑。
assert baseline_tests["subtract"] is False  # 执行本行的状态、计算或校验逻辑。
assert baseline_tests["zero"] is False  # 执行本行的状态、计算或校验逻辑。
assert all(isinstance(value, bool) for value in baseline_tests.values())  # 执行本行的状态、计算或校验逻辑。


## 4. 最小补丁：限制修改范围并保留证据

Agent loop 应先 inspect，再提出同基线绑定的 patch，再跑测试。代码审查常常还要求最小 diff、关联 issue、格式化和静态检查；此处只验证修改行数和唯一目标文件，不能把它误认为完整 code review。


In [ ]:
def changed_line_count(before, after):  # 执行本行的状态、计算或校验逻辑。
    left = before.splitlines()  # 执行本行的状态、计算或校验逻辑。
    right = after.splitlines()  # 执行本行的状态、计算或校验逻辑。
    return sum(a != b for a, b in zip(left, right)) + abs(len(left) - len(right))  # 执行本行的状态、计算或校验逻辑。
patch = Patch("calc.py", source, "def score(a, b):\n    return a - b\n")  # 执行本行的状态、计算或校验逻辑。
assert changed_line_count(patch.before, patch.after) == 1  # 执行本行的状态、计算或校验逻辑。
assert patch.path == "calc.py"  # 执行本行的状态、计算或校验逻辑。
assert patch.before != patch.after  # 执行本行的状态、计算或校验逻辑。


## 5. Agent loop：observe → patch → test → decide

最小 loop 的停止条件不能是模型自称完成，而应由测试 oracle 决定。若测试失败，应保留 trace 并进入下一轮受预算限制的修复；若通过，也应把 revision、patch 与测试结果打包。


In [ ]:
def repair_once(sandbox, patch, revision):  # 执行本行的状态、计算或校验逻辑。
    sandbox.apply(patch, revision)  # 执行本行的状态、计算或校验逻辑。
    report = run_tests(sandbox.files)  # 执行本行的状态、计算或校验逻辑。
    return {"passed": all(report.values()), "report": report, "revision": sandbox.revision}  # 执行本行的状态、计算或校验逻辑。
outcome = repair_once(sandbox, patch, revision)  # 执行本行的状态、计算或校验逻辑。
assert outcome["passed"] is True  # 执行本行的状态、计算或校验逻辑。
assert outcome["revision"] == 1  # 执行本行的状态、计算或校验逻辑。
assert sandbox.history == ["calc.py"]  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：陈旧 patch 必须拒绝

多 Agent 或长链思考时，观察和执行之间可能已有新提交。拒绝陈旧 patch 比静默覆盖更可控；调用者再重新 inspect 并生成新补丁。这个分支也能测试 Agent 是否真的遵从 sandbox 协议。


In [ ]:
stale = Patch("calc.py", source, "def score(a, b):\n    return 0\n")  # 执行本行的状态、计算或校验逻辑。
try:  # 执行本行的状态、计算或校验逻辑。
    sandbox.apply(stale, 0)  # 执行本行的状态、计算或校验逻辑。
    assert False  # 执行本行的状态、计算或校验逻辑。
except ValueError:  # 执行本行的状态、计算或校验逻辑。
    assert sandbox.revision == 1  # 执行本行的状态、计算或校验逻辑。
assert sandbox.files["calc.py"] == patch.after  # 执行本行的状态、计算或校验逻辑。


## 7. 评分：目标测试、回归与无关修改一起看

一个可靠的 grader 至少同时看目标测试、回归测试和变更范围。隐藏测试并不能证明程序完全正确，但能降低只针对公开样例过拟合的概率；结果还应绑定环境镜像与测试版本。


In [ ]:
def grade(sandbox, target_paths):  # 执行本行的状态、计算或校验逻辑。
    report = run_tests(sandbox.files)  # 执行本行的状态、计算或校验逻辑。
    path_ok = set(sandbox.history).issubset(set(target_paths))  # 执行本行的状态、计算或校验逻辑。
    return {"tests": all(report.values()), "scope": path_ok, "score": int(all(report.values()) and path_ok)}  # 执行本行的状态、计算或校验逻辑。
grade_report = grade(sandbox, ["calc.py"])  # 执行本行的状态、计算或校验逻辑。
assert grade_report["tests"] is True  # 执行本行的状态、计算或校验逻辑。
assert grade_report["scope"] is True  # 执行本行的状态、计算或校验逻辑。
assert grade_report["score"] == 1  # 执行本行的状态、计算或校验逻辑。


## 8. 制品版本：对齐 issue、基线和测试环境

线上交付要保存 issue 标识、基线 commit、patch hash、依赖镜像、测试结果和超时原因。没有这些字段，就无法判断失败是模型推理、环境漂移还是测试基础设施问题。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
artifact = {"issue": "demo-1", "base_revision": 0, "final_revision": sandbox.revision, "tests": "unit-v1"}  # 执行本行的状态、计算或校验逻辑。
patch_hash = hashlib.sha256(patch.after.encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["final_revision"] == 1  # 执行本行的状态、计算或校验逻辑。
assert len(patch_hash) == 64  # 执行本行的状态、计算或校验逻辑。
assert artifact["tests"] == "unit-v1"  # 执行本行的状态、计算或校验逻辑。


## 面试收束

按“任务目标 → 显式状态 → 动作前策略门禁 → 成功 oracle → 失败和重试 → 指标与版本化制品”的顺序回答。不要把一次文本看起来合理的演示当成可靠性证明：要独立检查状态、权限、不可逆副作用与多次运行的一致性。
